# AI Agents with LangGraph Part 1

In [ ]:
%pip install -qU langgraph langchain-google-genai

## Exercise 1: SQL Agents

SQL agents adalah AI Agent yang dapat mengambil data dari table SQL dan menganalisis sesuai dengan pertanyaan dari user.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os

from google.colab import userdata
GEMINI = userdata.get('GEMINI')
os.environ["GOOGLE_API_KEY"] = GEMINI

### Langkah 1: Define beberapa table SQL

In [ ]:
%load_ext sql
%sql sqlite:///library.db

In [ ]:
%%sql
PRAGMA foreign_keys = ON;

-- Where books are stored
CREATE TABLE IF NOT EXISTS locations (
    location_id INTEGER PRIMARY KEY AUTOINCREMENT,
    room        VARCHAR(100) NOT NULL,
    shelf       VARCHAR(50)  NOT NULL,
    UNIQUE (room, shelf)
);

-- Library staff
CREATE TABLE IF NOT EXISTS librarians (
    librarian_id INTEGER PRIMARY KEY AUTOINCREMENT,
    first_name   VARCHAR(255) NOT NULL,
    last_name    VARCHAR(255) NOT NULL
);

-- Library members
CREATE TABLE IF NOT EXISTS readers (
    reader_id  INTEGER PRIMARY KEY AUTOINCREMENT,
    first_name VARCHAR(255) NOT NULL,
    last_name  VARCHAR(255) NOT NULL,
    email      VARCHAR(255) UNIQUE,
    phone      VARCHAR(50)
);

-- Books, with their location and current status
CREATE TABLE IF NOT EXISTS books (
    book_id        INTEGER PRIMARY KEY AUTOINCREMENT,
    title          VARCHAR(255) NOT NULL,
    author         VARCHAR(255) NOT NULL,
    isbn           VARCHAR(20) UNIQUE,
    published_year INTEGER,
    location_id    INTEGER NOT NULL,
    status         VARCHAR(20) NOT NULL DEFAULT 'available'
                   CHECK (status IN ('available', 'borrowed')),
    FOREIGN KEY (location_id) REFERENCES locations (location_id)
);

-- Borrowing history: who borrowed which book, handled by which librarian
CREATE TABLE IF NOT EXISTS loans (
    loan_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    book_id      INTEGER NOT NULL,
    reader_id    INTEGER NOT NULL,
    librarian_id INTEGER NOT NULL,
    borrow_date  DATE NOT NULL DEFAULT (DATE('now')),
    due_date     DATE NOT NULL,
    return_date  DATE,                -- NULL = not returned yet
    FOREIGN KEY (book_id)      REFERENCES books (book_id),
    FOREIGN KEY (reader_id)    REFERENCES readers (reader_id),
    FOREIGN KEY (librarian_id) REFERENCES librarians (librarian_id)
);

-- A book can only have one open (unreturned) loan at a time
CREATE UNIQUE INDEX IF NOT EXISTS one_open_loan_per_book
    ON loans (book_id) WHERE return_date IS NULL;

-- Sample data
INSERT INTO locations (room, shelf) VALUES
    ('Main Hall', 'A1'),
    ('Main Hall', 'A2'),
    ('Reading Room', 'B1');

INSERT INTO librarians (first_name, last_name) VALUES
    ('Alice', 'Smith'),
    ('Bob', 'Johnson');

INSERT INTO readers (first_name, last_name, email, phone) VALUES
    ('David', 'Lee',   'david@example.com', '0811-111-111'),
    ('Emily', 'Chen',  'emily@example.com', '0812-222-222'),
    ('Frank', 'Brown', 'frank@example.com', '0813-333-333');

INSERT INTO books (title, author, isbn, published_year, location_id) VALUES
    ('Clean Code',                          'Robert C. Martin',           '9780132350884', 2008, 1),
    ('The Pragmatic Programmer',            'Hunt & Thomas',              '9780135957059', 2019, 1),
    ('Database System Concepts',            'Silberschatz',               '9780078022159', 2019, 2),
    ('Laskar Pelangi',                      'Andrea Hirata',              '9789793062792', 2005, 3),
    ('The Answer',                          'Anderson Jeremy',            '9780000000001', 2005, 3),
    ('Refactoring',                         'Martin Fowler',              '9780000000002', 2018, 1),
    ('Design Patterns',                     'Gamma, Helm, Johnson, Vlissides', '9780000000003', 1994, 1),
    ('Introduction to Algorithms',          'Cormen et al.',              '9780000000004', 2009, 2),
    ('Code Complete',                       'Steve McConnell',            '9780000000005', 2004, 1),
    ('The Mythical Man-Month',              'Fred Brooks',                '9780000000006', 1975, 2),
    ('Structure and Interpretation of Computer Programs', 'Abelson & Sussman', '9780000000007', 1996, 2),
    ('Eloquent JavaScript',                 'Marijn Haverbeke',           '9780000000008', 2018, 1),
    ('Python Crash Course',                 'Eric Matthes',               '9780000000009', 2019, 1),
    ('Designing Data-Intensive Applications','Martin Kleppmann',          '9780000000010', 2017, 2),
    ('SQL Antipatterns',                    'Bill Karwin',                '9780000000011', 2010, 2),
    ('Bumi Manusia',                        'Pramoedya Ananta Toer',      '9780000000012', 1980, 3),
    ('Anak Semua Bangsa',                   'Pramoedya Ananta Toer',      '9780000000013', 1980, 3),
    ('Ronggeng Dukuh Paruk',                'Ahmad Tohari',               '9780000000014', 1982, 3),
    ('Cantik Itu Luka',                     'Eka Kurniawan',              '9780000000015', 2002, 3),
    ('Negeri 5 Menara',                     'Ahmad Fuadi',                '9780000000016', 2009, 3),
    ('Sang Pemimpi',                        'Andrea Hirata',              '9780000000017', 2006, 3),
    ('Ayat-Ayat Cinta',                     'Habiburrahman El Shirazy',   '9780000000018', 2004, 3),
    ('Pulang',                              'Leila S. Chudori',           '9780000000019', 2012, 3),
    ('Sapiens',                             'Yuval Noah Harari',          '9780000000020', 2011, 2),
    ('Atomic Habits',                       'James Clear',                '9780000000021', 2018, 2);

-- David borrows book 1 (handled by Alice), Emily borrows book 3 (handled by Bob)
INSERT INTO loans (book_id, reader_id, librarian_id, borrow_date, due_date) VALUES
    (1, 1, 1, '2026-10-20', '2026-11-03'),
    (3, 2, 2, '2026-10-22', '2026-11-05');

UPDATE books SET status = 'borrowed' WHERE book_id IN (1, 3);

In [ ]:
import sqlite3

db_file = "library.db"

### Langkah 2: Define Tools
Tools yang akan digunakan pada SQL agent adalah `list_tables`, `describe_tables`, dan `execute_query` sehingga AI dapat memahami metadata table terlebih dahulu sebelum mulai menjalankan query

In [ ]:
def list_tables() -> list[str]:
    """Retrieve the names of all tables in the database."""
    # Include print logging statements so you can see when functions are being called.
    print(' - DB CALL: list_tables')

    # Create a new connection for thread safety
    with sqlite3.connect(db_file) as conn:
        cursor = conn.cursor()

        # Fetch the table names.
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")

        tables = cursor.fetchall()
        return [t[0] for t in tables]


list_tables()

In [ ]:
def describe_table(table_name: str) -> list[tuple[str, str]]:
    """Look up the table schema.

    Returns:
      List of columns, where each entry is a tuple of (column, type).
    """
    print(' - DB CALL: describe_table')

    # Create a new connection for thread safety
    with sqlite3.connect(db_file) as conn:
        cursor = conn.cursor()

        cursor.execute(f"PRAGMA table_info({table_name});")

        schema = cursor.fetchall()
        # [column index, column name, column type, ...]
        return [(col[1], col[2]) for col in schema]


describe_table("books")

In [ ]:
def execute_query(sql: str) -> list[list[str]]:
    """Execute a SELECT statement, returning the results."""
    print(' - DB CALL: execute_query')

    # Create a new connection for thread safety
    with sqlite3.connect(db_file) as conn:
        cursor = conn.cursor()

        cursor.execute(sql)
        return cursor.fetchall()


execute_query("select * from books")

### Langkah 3: Bangun chatbot dengan create_react_agent

In [ ]:
# These are the Python functions defined above.
db_tools = [list_tables, describe_table, execute_query]

instruction = """
# Role
You are "Libby", a friendly library assistant for a public library. You help
readers and librarians find books, check where a book is shelved, see whether
it is available or borrowed, and who borrowed it.

# Tone
- Warm, polite and concise. Use simple language.
- Answer in the same language the user writes in (English or Bahasa Indonesia).
- When listing several books, use a short bulleted list.

# How to use the database
- ALWAYS start by calling list_tables to discover the available tables.
- ALWAYS call describe_table on the relevant tables before writing SQL.
- Never assume or guess table or column names; verify first.
- Only run SELECT queries. Never INSERT, UPDATE, DELETE or DROP anything.

# Privacy
- Do not reveal readers' email or phone numbers unless the user says they are a librarian.

# If you can't find it
- Say so honestly and suggest a similar search (e.g. by author or partial title).
"""

# Import the necessary modules from langgraph
from langchain.agents import create_agent

# Initialize the chat model with specific parameters
model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0
)

# Create a ReAct agent using the model, tools, and prompt
agent = create_agent(
    model=model,
    tools=db_tools,
    system_prompt=instruction
)

## Setup: Install Library & Konfigurasi ngrok

### Step 1 — Install Streamlit dan pyngrok

In [ ]:
!pip install -q streamlit pyngrok google-genai

### Step 2 — Daftarkan ngrok Auth Token

Cara mendapatkan token:
1. Daftar gratis di [https://ngrok.com](https://ngrok.com)
2. Login → klik **Your Authtoken** di dashboard
3. Copy token-nya, paste ke Colab Secrets dengan nama `NGROK_TOKEN`
   (klik ikon 🔑 di sidebar kiri Colab)

> Token ngrok gratis sudah cukup untuk keperluan training.
> Satu akun ngrok bisa membuka 1 tunnel aktif sekaligus.

In [ ]:
from pyngrok import ngrok
from pyngrok.exception import PyngrokNgrokError
from google.colab import userdata

# Ambil token dari Colab Secrets
try:
    ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))
    print("ngrok token berhasil dikonfigurasi!")
except PyngrokNgrokError as e:
    if "already set" in str(e):
        print("ngrok token sudah terkonfigurasi sebelumnya.")
    else:
        raise e

### Fungsi Helper: Jalankan Streamlit + Buka Tunnel

Kita buat fungsi reusable yang akan kita pakai berulang kali di setiap bagian.
Fungsi ini melakukan tiga hal:
1. Menulis file `.py` ke disk
2. Menjalankan `streamlit run` di background
3. Membuka tunnel ngrok dan menampilkan URL publik

In [ ]:
import subprocess
import time

def run_streamlit(filename, port=8501):
    # Kill SEMUA proses streamlit, bukan hanya yang kita spawn
    subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)

    # Force-free port kalau masih ada yang nempel
    subprocess.run(["fuser", "-k", f"{port}/tcp"], capture_output=True)

    # Tutup semua tunnel ngrok
    ngrok.kill()

    # Tunggu port benar-benar bebas
    time.sleep(3)

    proc = subprocess.Popen(
        [
            "streamlit", "run", filename,
            "--server.headless=true",
            "--server.port", str(port),
            "--server.enableCORS=false",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    time.sleep(3)

    public_url = ngrok.connect(port)
    print(f"Streamlit berjalan: {public_url}")

    return proc

---

# Part 2: Chatbot dengan Gemini

Sekarang kita masuk ke bagian utama — membangun chatbot berbasis LLM menggunakan Streamlit + Gemini API.
Sumber: `streamlit_chat_app.py` dari repo demo.

## Konsep Kunci: st.session_state

Ini adalah konsep paling penting untuk memahami cara kerja chatbot di Streamlit.

Ingat bahwa Streamlit menjalankan ulang seluruh skrip setiap kali ada interaksi.
Ini artinya semua variabel Python akan di-reset ke nilai awal setiap kali user kirim pesan.

**Masalah:** Kalau pesan-pesan chat disimpan di variabel biasa, semua pesan akan hilang setiap kali user mengetik sesuatu!

**Solusi:** `st.session_state` adalah dictionary khusus yang **nilainya tetap tersimpan** meskipun skrip dijalankan ulang.

```python
# Variabel biasa — akan HILANG setiap rerun
messages = []

# st.session_state — akan BERTAHAN setiap rerun  
st.session_state.messages = []
```

Di chatbot kita, `st.session_state` digunakan untuk menyimpan:
- `messages` — riwayat semua pesan (user + assistant)
- `chat` — objek chat session dari Gemini API (menyimpan konteks percakapan)
- `genai_client` — client Gemini yang sudah terinisialisasi
- `_last_key` — API key terakhir yang dipakai (untuk deteksi perubahan key)

## Komponen Chat Khusus Streamlit

```python
# Menampilkan bubble chat — role bisa "user" atau "assistant"
with st.chat_message("user"):
    st.markdown("pesan dari user")

with st.chat_message("assistant"):
    st.markdown("jawaban dari AI")

# Kotak input di bagian bawah halaman
prompt = st.chat_input("Ketik pesanmu...")
```

## Alur Logic Chatbot

```
Skrip dijalankan (setiap interaksi)
         ↓
Cek API key di sidebar
         ↓
Inisialisasi client & chat session (hanya kalau belum ada di session_state)
         ↓
Tampilkan semua pesan dari st.session_state.messages
         ↓
Tunggu input dari st.chat_input()
         ↓
User kirim pesan?
   ↓ Ya
Tambah pesan user ke messages
Tampilkan bubble user
Kirim ke Gemini API
Tampilkan bubble assistant
Tambah jawaban ke messages
   ↓
Kembali ke atas (skrip dijalankan ulang)
```

## Cara Mendapatkan Google AI API Key

1. Buka [https://aistudio.google.com](https://aistudio.google.com)
2. Klik **Get API Key** → **Create API Key**
3. Copy key-nya
4. Di app Streamlit nanti, paste langsung ke kotak "Google AI API Key" di sidebar

> Key tidak perlu disimpan di Colab Secrets — user memasukkannya langsung di UI app.

In [ ]:
%%writefile streamlit_chat_app.py
# Import library yang dibutuhkan
import os
import sqlite3
import streamlit as st
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

# Lokasi database (file yang sama dengan yang dibuat di Langkah 1)
DB_FILE = os.path.join(os.path.dirname(os.path.abspath(__file__)), "library.db")

# ── 1. Konfigurasi Halaman ───────────────────────────────────────────────────
st.title("📚 Libby – Library Agent")
st.caption("Chatbot perpustakaan yang bisa membaca data buku yang ada di perpustakaan dan ketersediaannya untuk bisa dipinjam dan lokasi buku")

# ── 2. Sidebar: Pengaturan App ───────────────────────────────────────────────
with st.sidebar:
    st.subheader("Pengaturan")
    google_api_key = st.text_input("Google AI API Key", type="password")
    show_tools = st.checkbox("Tampilkan tool calls", value=True,
                             help="Lihat tool apa saja yang dipanggil agent")
    reset_button = st.button("Reset Percakapan", help="Hapus semua pesan dan mulai dari awal")

# ── 3. Validasi API Key & Database ───────────────────────────────────────────
if not google_api_key:
    st.info("Masukkan Google AI API Key di sidebar untuk mulai chat.", icon="🗝️")
    st.stop()

if not os.path.exists(DB_FILE):
    st.error(f"Database tidak ditemukan: {DB_FILE}. Jalankan cell %%sql di Langkah 1 dulu.")
    st.stop()

# ── 4. Tools untuk SQL Agent (sama seperti Langkah 2) ────────────────────────
# Streamlit berjalan di proses terpisah, jadi tools harus didefinisikan ulang di sini.
# Koneksi dibuka read-only supaya agent tidak bisa mengubah data.
def _connect():
    return sqlite3.connect(f"file:{DB_FILE}?mode=ro", uri=True)

def list_tables() -> list[str]:
    """Retrieve the names of all tables in the database."""
    with _connect() as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        return [t[0] for t in cursor.fetchall()]

def describe_table(table_name: str) -> list[tuple[str, str]]:
    """Look up the table schema.

    Returns:
      List of columns, where each entry is a tuple of (column, type).
    """
    with _connect() as conn:
        cursor = conn.cursor()
        cursor.execute(f"PRAGMA table_info({table_name});")
        return [(col[1], col[2]) for col in cursor.fetchall()]

def execute_query(sql: str) -> list[list[str]]:
    """Execute a SELECT statement, returning the results."""
    with _connect() as conn:
        cursor = conn.cursor()
        cursor.execute(sql)
        return cursor.fetchall()

db_tools = [list_tables, describe_table, execute_query]

# ── 5. System Prompt (sama seperti Langkah 3) ────────────────────────────────
instruction = """
# Role
You are "Libby", a friendly library assistant for a public library. You help
readers and librarians find books, check where a book is shelved, see whether
it is available or borrowed, and who borrowed it.

# Tone
- Warm, polite and concise. Use simple language.
- Answer in the same language the user writes in (English or Bahasa Indonesia).
- When listing several books, use a short bulleted list.

# How to use the database
- ALWAYS start by calling list_tables to discover the available tables.
- ALWAYS call describe_table on the relevant tables before writing SQL.
- Never assume or guess table or column names; verify first.
- Only run SELECT queries. Never INSERT, UPDATE, DELETE or DROP anything.

# Privacy
- Do not reveal readers' email or phone numbers unless the user says they are a librarian.

# If you can't find it
- Say so honestly and suggest a similar search (e.g. by author or partial title).
"""

# ── 6. Inisialisasi Agent ────────────────────────────────────────────────────
# Agent hanya dibuat ulang kalau belum ada atau API key berganti
if ("agent" not in st.session_state) or (
    st.session_state.get("_last_key") != google_api_key
):
    try:
        model = ChatGoogleGenerativeAI(
            model="gemini-3.5-flash-lite",
            temperature=0,
            google_api_key=google_api_key,
        )
        st.session_state.agent = create_agent(
            model=model,
            tools=db_tools,
            system_prompt=instruction,
        )
        st.session_state._last_key = google_api_key
        st.session_state.pop("messages", None)
    except Exception as e:
        st.error(f"Gagal membuat agent: {e}")
        st.stop()

if "messages" not in st.session_state:
    st.session_state.messages = []

# ── 7. Tombol Reset ──────────────────────────────────────────────────────────
if reset_button:
    st.session_state.pop("messages", None)
    st.rerun()

# ── Helper: ambil teks dari konten pesan Gemini ──────────────────────────────
# Gemini kadang mengembalikan content berupa list of parts, bukan string
def to_text(content) -> str:
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for p in content:
            if isinstance(p, str):
                parts.append(p)
            elif isinstance(p, dict) and p.get("type") == "text":
                parts.append(p.get("text", ""))
        return "".join(parts)
    return str(content)

# ── 8. Tampilkan Riwayat Percakapan ──────────────────────────────────────────
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# ── 9. Input & Respons ───────────────────────────────────────────────────────
prompt = st.chat_input("Tanya Libby, misal: 'Buku apa saja karya Andrea Hirata?'")

if prompt:
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        try:
            with st.spinner("Libby sedang mencari di database..."):
                # Kirim seluruh riwayat ke agent supaya konteks percakapan terjaga
                history = [{"role": m["role"], "content": m["content"]}
                           for m in st.session_state.messages]
                result = st.session_state.agent.invoke({"messages": history})

            new_msgs = result["messages"][len(history):]

            # Tampilkan tool calls (opsional) agar terlihat proses berpikir agent
            if show_tools:
                calls = [tc for m in new_msgs for tc in (getattr(m, "tool_calls", None) or [])]
                if calls:
                    with st.expander(f"🔧 {len(calls)} tool call"):
                        for tc in calls:
                            st.code(f"{tc['name']}({tc['args']})", language="python")

            answer = to_text(result["messages"][-1].content) or "_(tidak ada jawaban)_"
        except Exception as e:
            answer = f"Terjadi error: {e}"

        st.markdown(answer)

    st.session_state.messages.append({"role": "assistant", "content": answer})

Jalankan chatbot-nya:

In [ ]:
# Putuskan semua koneksi ngrok yang masih menggantung secara paksa sebelum menjalankan
try:
    for t in ngrok.get_tunnels():
        ngrok.disconnect(t.public_url)
    ngrok.kill()
except Exception:
    pass

# Jalankan streamlit kembali
proc = run_streamlit("streamlit_chat_app.py")

---

## Menghentikan App

Jalankan cell di bawah untuk menghentikan Streamlit dan menutup tunnel ngrok.

In [ ]:
# Hentikan Streamlit
try:
    proc.terminate()
    print("Streamlit dihentikan.")
except:
    print("Tidak ada proses yang berjalan.")

# Tutup semua tunnel ngrok
ngrok.kill()
print("Tunnel ngrok ditutup.")